# RMA Summary of Business — AIP Sales Dashboard

Interactive sales report built on the USDA RMA **Summary of Business (SOB)** `YYsumdat` files.

**How it works**
* All extraction, parsing and cleaning lives in `data_pipeline.py`. This notebook only loads the processed output and renders the dashboard.
* Run the notebook top to bottom. The default view shows **all sales data combined**; use the dropdowns to narrow to a commodity, insurance plan, AIP, year or state. Every KPI, chart and table updates on change.

**About the AIP filter:** the public SOB files have no AIP/company field. Unless you built the data with `--aip-file`, the AIP dropdown filters by *delivery channel* (Reinsured = AIP-delivered vs. Federal). The banner below tells you which mode is active.

Requirements: `pip install pandas requests pyarrow python-docx plotly ipywidgets`

## 1. Setup

In [ ]:
# --- Configuration -------------------------------------------------------------
BUILD_IF_MISSING = True   # run the pipeline automatically if no processed data exists
FORCE_REBUILD    = False  # re-run the pipeline even if processed data exists
PIPELINE_YEARS   = None   # e.g. [2025, 2026]; None = latest complete crop year
PIPELINE_ALL     = False  # True = every year via allsumdat.zip (large download)
AIP_FILE         = None   # optional AIP-level extract, see data_pipeline.load_aip_extract
TOP_N            = 15     # bars shown in the ranking charts

import logging
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import display, HTML, clear_output

import data_pipeline as dp

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
pd.options.display.float_format = "{:,.0f}".format

In [ ]:
# --- Load processed data (built by data_pipeline.py) ----------------------------
if FORCE_REBUILD or (BUILD_IF_MISSING and not dp.META_JSON.exists()):
    dp.run_pipeline(years=PIPELINE_YEARS, all_years=PIPELINE_ALL, aip_file=AIP_FILE)

df, meta = dp.load_processed()

aip_mode = meta.get("aip_source") == "aip_extract"
AIP_LABEL = "AIP" if aip_mode else "AIP / Delivery channel"
banner_color = "#1b7f4b" if aip_mode else "#a15c00"
banner_text = ("AIP filter uses your AIP-level extract: <b>{}</b>".format(meta.get("aip_file"))
               if aip_mode else
               "Public SOB data has no AIP field, so the AIP filter shows <b>delivery channel</b> "
               "(Reinsured = AIP-delivered). Rebuild with <code>--aip-file</code> to filter by company.")
display(HTML(f'''<div style="border-left:4px solid {banner_color};padding:8px 12px;margin:4px 0;background:rgba(127,127,127,.08)">
<b>Data:</b> crop year(s) {", ".join(map(str, meta["crop_years"]))} · {meta["rows"]:,} records ·
layout {meta["layout_source"]} · names from {meta["names_source"]} · built {meta["built_at"]}<br>{banner_text}</div>'''))

## 2. Metric definitions & aggregation helpers

In [ ]:
METRICS = {  # column -> display label
    "total_prem": "Total Premium ($)",
    "liability_amt": "Liability ($)",
    "subsidy": "Subsidy ($)",
    "farmer_paid_prem": "Farmer-paid Premium ($)",
    "indem_amt": "Indemnity ($)",
    "pol_sold_cnt": "Policies Sold",
    "pol_prem_cnt": "Policies Earning Premium",
    "net_acre_qty": "Net Acres",
}

def summarize(frame: pd.DataFrame, by=None) -> pd.DataFrame:
    """Sum the metrics (optionally grouped) and add ratio metrics computed AFTER aggregation."""
    cols = list(METRICS)
    agg = frame[cols].sum().to_frame().T if by is None else frame.groupby(by, observed=True)[cols].sum().reset_index()
    prem = agg["total_prem"].where(agg["total_prem"] != 0)
    agg["loss_ratio"] = agg["indem_amt"] / prem
    agg["subsidy_pct"] = agg["subsidy"] / prem
    agg["prem_rate"] = agg["total_prem"] / agg["liability_amt"].where(agg["liability_amt"] != 0)
    return agg

def fmt_money(v):
    for div, suf in ((1e9, "B"), (1e6, "M"), (1e3, "K")):
        if abs(v) >= div:
            return f"${v/div:,.2f}{suf}"
    return f"${v:,.0f}"

def kpi_cards(frame: pd.DataFrame) -> str:
    s = summarize(frame).iloc[0]
    cards = [
        ("Total Premium", fmt_money(s.total_prem)),
        ("Liability", fmt_money(s.liability_amt)),
        ("Policies Sold", f"{s.pol_sold_cnt:,.0f}"),
        ("Net Acres", f"{s.net_acre_qty:,.0f}"),
        ("Subsidy", f"{fmt_money(s.subsidy)} ({s.subsidy_pct:.0%})" if pd.notna(s.subsidy_pct) else fmt_money(s.subsidy)),
        ("Indemnity", fmt_money(s.indem_amt)),
        ("Loss Ratio", f"{s.loss_ratio:.2f}" if pd.notna(s.loss_ratio) else "–"),
    ]
    html = "".join(f'<div style="flex:1 1 140px;padding:10px 14px;border:1px solid rgba(127,127,127,.3);border-radius:8px">'
                   f'<div style="font-size:12px;opacity:.7">{k}</div><div style="font-size:20px;font-weight:600">{v}</div></div>'
                   for k, v in cards)
    return f'<div style="display:flex;flex-wrap:wrap;gap:10px;margin:6px 0 12px">{html}</div>'

## 3. Filters

In [ ]:
ALL = "(All)"

def options(col, frame=df):
    return [ALL] + sorted(frame[col].dropna().astype(str).unique().tolist())

dd_layout = w.Layout(width="360px")
style = {"description_width": "150px"}
f_year  = w.Dropdown(options=[ALL] + [str(y) for y in sorted(df["crop_yr"].unique())], description="Crop Year", layout=dd_layout, style=style)
f_comm  = w.Dropdown(options=options("commodity"), description="Commodity", layout=dd_layout, style=style)
f_plan  = w.Dropdown(options=options("insurance_plan"), description="Insurance Plan", layout=dd_layout, style=style)
f_aip   = w.Dropdown(options=options("aip"), description=AIP_LABEL, layout=dd_layout, style=style)
f_state = w.Dropdown(options=options("state_abbr"), description="State", layout=dd_layout, style=style)
f_metric = w.ToggleButtons(options=[("Premium", "total_prem"), ("Liability", "liability_amt"),
                                    ("Policies Sold", "pol_sold_cnt"), ("Net Acres", "net_acre_qty")],
                           description="Chart metric", style={"description_width": "90px"})
b_reset = w.Button(description="Reset filters", icon="refresh")

FILTERS = {"crop_yr": f_year, "commodity": f_comm, "insurance_plan": f_plan, "aip": f_aip, "state_abbr": f_state}

def filtered(exclude=None) -> pd.DataFrame:
    """Apply every active filter (optionally skipping one, used to build cascading options)."""
    mask = pd.Series(True, index=df.index)
    for col, widget in FILTERS.items():
        if col != exclude and widget.value != ALL:
            mask &= df[col].astype(str) == widget.value
    return df[mask]

_updating = False
def refresh_options():
    """Cascade: each dropdown only lists values that exist under the *other* active filters."""
    global _updating
    _updating = True
    try:
        for col, widget in FILTERS.items():
            current = widget.value
            opts = options(col, filtered(exclude=col))
            widget.options = opts
            widget.value = current if current in opts else ALL
    finally:
        _updating = False

## 4. Charts

In [ ]:
def bar_rank(frame, by, metric, title):
    agg = summarize(frame, by).nlargest(TOP_N, metric).sort_values(metric)
    fig = px.bar(agg, x=metric, y=by, orientation="h", title=title,
                 labels={metric: METRICS[metric], by: ""},
                 hover_data={"loss_ratio": ":.2f", "total_prem": ":,.0f", "liability_amt": ":,.0f"})
    fig.update_layout(height=max(320, 26 * len(agg) + 110), margin=dict(l=10, r=10, t=50, b=10))
    return fig

def state_map(frame, metric):
    agg = summarize(frame, "state_abbr")
    fig = px.choropleth(agg, locations="state_abbr", locationmode="USA-states", color=metric, scope="usa",
                        color_continuous_scale="Greens", labels={metric: METRICS[metric]},
                        title=f"{METRICS[metric]} by State", hover_data={"loss_ratio": ":.2f"})
    fig.update_layout(height=420, margin=dict(l=10, r=10, t=50, b=10))
    return fig

def coverage_chart(frame, metric):
    agg = summarize(frame, ["cov_lvl_pct", "coverage_type"])
    agg = agg[agg[metric] > 0]
    fig = px.bar(agg, x="cov_lvl_pct", y=metric, color="coverage_type",
                 labels={"cov_lvl_pct": "Coverage level (%)", metric: METRICS[metric], "coverage_type": "Coverage"},
                 title=f"{METRICS[metric]} by Coverage Level")
    fig.update_layout(height=360, margin=dict(l=10, r=10, t=50, b=10), bargap=0.15)
    return fig

def channel_chart(frame, metric):
    agg = summarize(frame, "aip").sort_values(metric, ascending=False)
    if len(agg) > 8:  # many real AIPs -> bar; few channels -> donut
        return bar_rank(frame, "aip", metric, f"{METRICS[metric]} by {AIP_LABEL}")
    fig = px.pie(agg, names="aip", values=metric, hole=0.5, title=f"{METRICS[metric]} by {AIP_LABEL}")
    fig.update_layout(height=360, margin=dict(l=10, r=10, t=50, b=10))
    return fig

def year_trend(frame):
    agg = summarize(frame, "crop_yr").sort_values("crop_yr")
    fig = go.Figure()
    fig.add_bar(x=agg["crop_yr"], y=agg["total_prem"], name="Total Premium")
    fig.add_bar(x=agg["crop_yr"], y=agg["indem_amt"], name="Indemnity")
    fig.add_scatter(x=agg["crop_yr"], y=agg["loss_ratio"], name="Loss ratio", yaxis="y2", mode="lines+markers")
    fig.update_layout(title="Premium, Indemnity & Loss Ratio by Crop Year", barmode="group", height=380,
                      yaxis=dict(title="$"), yaxis2=dict(title="Loss ratio", overlaying="y", side="right"),
                      margin=dict(l=10, r=10, t=50, b=10), legend=dict(orientation="h", y=-0.15))
    return fig

def summary_table(frame) -> pd.DataFrame:
    t = summarize(frame, ["commodity", "insurance_plan"]).sort_values("total_prem", ascending=False).head(25)
    t = t.rename(columns={**METRICS, "commodity": "Commodity", "insurance_plan": "Plan",
                          "loss_ratio": "Loss Ratio", "subsidy_pct": "Subsidy %", "prem_rate": "Premium Rate"})
    return (t.style.format({c: "{:,.0f}" for c in METRICS.values()})
             .format({"Loss Ratio": "{:.2f}", "Subsidy %": "{:.0%}", "Premium Rate": "{:.2%}"}, na_rep="–")
             .hide(axis="index"))

## 5. Dashboard

In [ ]:
out = w.Output()

def render(*_):
    if _updating:
        return
    refresh_options()
    sub = filtered()
    metric = f_metric.value
    active = [f"{w_.description}: <b>{w_.value}</b>" for w_ in FILTERS.values() if w_.value != ALL]
    with out:
        clear_output(wait=True)
        display(HTML(f"<h3 style='margin:4px 0'>{' · '.join(active) if active else 'All sales data combined'}</h3>"
                     f"<div style='opacity:.7'>{len(sub):,} records</div>"))
        if sub.empty:
            display(HTML("<p>No records match these filters.</p>"))
            return
        display(HTML(kpi_cards(sub)))
        if sub["crop_yr"].nunique() > 1:
            year_trend(sub).show()
        bar_rank(sub, "commodity", metric, f"Top {TOP_N} Commodities by {METRICS[metric]}").show()
        bar_rank(sub, "insurance_plan", metric, f"Top {TOP_N} Insurance Plans by {METRICS[metric]}").show()
        channel_chart(sub, metric).show()
        state_map(sub, metric).show()
        coverage_chart(sub, metric).show()
        display(HTML("<h4>Top products (commodity × plan) by premium</h4>"))
        display(summary_table(sub))

def reset(_):
    global _updating
    _updating = True
    for widget in FILTERS.values():
        widget.value = ALL
    _updating = False
    render()

for widget in [*FILTERS.values(), f_metric]:
    widget.observe(render, names="value")
b_reset.on_click(reset)

controls = w.VBox([
    w.HBox([f_year, f_state]),
    w.HBox([f_comm, f_plan]),
    w.HBox([f_aip, b_reset]),
    f_metric,
])
display(controls, out)
render()  # default view: everything combined

## 6. Export the current view (optional)

In [ ]:
# Saves the currently filtered subset and its commodity × plan summary to CSV.
sub = filtered()
summarize(sub, ["crop_yr", "commodity", "insurance_plan", "aip"]).to_csv("sob_current_view_summary.csv", index=False)
print(f"Saved {len(sub):,} filtered records summarized to sob_current_view_summary.csv")

## Notes & caveats
* **Dollar fields** are whole dollars as reported by RMA; **net acres** are reported quantity × insured share (for non-acreage crops the unit may not be acres).
* **Ratios** (loss ratio, subsidy %, premium rate) are computed after aggregation, never averaged across rows.
* **Early-season years:** the newest crop year on the SOB site is usually partial (fall-planted crops only). Compare it to prior years with care.
* **Code mappings** for `coverage_flag` and `delivery_sys` in `data_pipeline.py` follow RMA's sobcov documentation; check a few values against RMA's published totals before relying on them.
* **AIP-level analysis** needs data outside the public SOB. Provide it with `python data_pipeline.py --aip-file <file>`; the pipeline also writes `data/processed/aip_reconciliation.csv` comparing your extract to public reinsured totals.